In [ ]:
import sys
sys.path.insert(0, "..")

# Data Exploration

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from datetime import datetime
from warnings import filterwarnings
import logging

matplotlib.rc("font", **{"size": 14})
logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
filterwarnings("ignore")

from src.data.option_db import SPYOptionLoader, AAPLOptionLoader, extract_spot_from_options
from src.data.rates_db import USRatesLoader
from src.trading.selection import select_options, select_closest_maturity

## 1. Loading the Data

In [ ]:
START_DATE = datetime(2020, 1, 2)
END_DATE   = datetime(2022, 12, 30)

df_spy = SPYOptionLoader.load_data(
    start_date=START_DATE,
    end_date=END_DATE,
    process_kwargs={"ticker": "SPY"},
)
df_spy.head()

In [ ]:
df_aapl = AAPLOptionLoader.load_data(
    start_date=START_DATE,
    end_date=END_DATE,
    process_kwargs={"ticker": "AAPL"},
)
df_aapl.head()

In [ ]:
df_rates = USRatesLoader.load_data(start_date=START_DATE, end_date=END_DATE)
df_rates.head()

In [ ]:
print(f"SPY options : {df_spy.shape}   | dates: {df_spy['date'].min().date()} → {df_spy['date'].max().date()}")
print(f"AAPL options: {df_aapl.shape}  | dates: {df_aapl['date'].min().date()} → {df_aapl['date'].max().date()}")
print(f"US Rates    : {df_rates.shape} | dates: {df_rates['date'].min().date()} → {df_rates['date'].max().date()}")

## 2. Spot Prices

In [ ]:
df_spy_spot  = extract_spot_from_options(df_spy)
df_aapl_spot = extract_spot_from_options(df_aapl)

fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(df_spy_spot["date"],  df_spy_spot["spot"],  label="SPY",  color="steelblue")
axes[0].set_title("SPY Spot Price"); axes[0].set_ylabel("Price ($)"); axes[0].legend()
axes[1].plot(df_aapl_spot["date"], df_aapl_spot["spot"], label="AAPL", color="darkorange")
axes[1].set_title("AAPL Spot Price"); axes[1].set_ylabel("Price ($)"); axes[1].legend()
fig.suptitle("Spot Prices 2020–2022"); plt.tight_layout(); plt.show()

## 3. Implied Volatility — Surface Snapshots

In [ ]:
REF_DATES = [datetime(2020, 1, 15), datetime(2020, 3, 20), datetime(2021, 6, 15)]
MATURITY_TARGETS = [7, 30, 60, 90]

for ref_date in REF_DATES:
    fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
    for ax, (df, ticker) in zip(axes, [(df_spy, "SPY"), (df_aapl, "AAPL")]):
        df_day = df[(df["date"] == ref_date) & (df["call_put"] == "C")].copy()
        for dte in MATURITY_TARGETS:
            bucket = df_day[abs(df_day["day_to_expiration"] - dte) <= 10].sort_values("moneyness")
            if len(bucket) > 2:
                ax.plot(bucket["moneyness"], bucket["implied_volatility"] * 100,
                        label=f"~{dte}d", marker="o", markersize=3)
        ax.set_title(f"{ticker} — IV smile {ref_date.date()}")
        ax.set_xlabel("Moneyness (K/S)"); ax.set_ylabel("Implied Vol (%)"); ax.legend()
    plt.tight_layout(); plt.show()

## 4. ATM IV Term Structure Over Time

In [ ]:
df_spy_atm_30 = select_options(
    df_spy,
    call_or_put="C",
    strike_col="moneyness",
    strike_target=1.0,
    day_to_expiry_target=30,
)

df_aapl_atm_30 = select_options(
    df_aapl,
    call_or_put="C",
    strike_col="moneyness",
    strike_target=1.0,
    day_to_expiry_target=30,
)

fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(df_spy_atm_30["date"],  df_spy_atm_30["implied_volatility"]  * 100, label="SPY ATM 30d IV",  color="steelblue")
ax.plot(df_aapl_atm_30["date"], df_aapl_atm_30["implied_volatility"] * 100, label="AAPL ATM 30d IV", color="darkorange")
ax.set_title("ATM 30-day Implied Volatility — SPY vs AAPL")
ax.set_ylabel("Implied Vol (%)"); ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
df_iv_spread = df_spy_atm_30[["date"]].copy().set_index("date")
df_iv_spread["spy_iv"]  = df_spy_atm_30.set_index("date")["implied_volatility"].values
df_iv_spread["aapl_iv"] = df_aapl_atm_30.set_index("date")["implied_volatility"].reindex(df_iv_spread.index)
df_iv_spread["iv_spread"] = (df_iv_spread["aapl_iv"] - df_iv_spread["spy_iv"]) * 100

fig, ax = plt.subplots(figsize=(14, 4))
ax.fill_between(df_iv_spread.index, df_iv_spread["iv_spread"], alpha=0.4, color="purple")
ax.plot(df_iv_spread.index, df_iv_spread["iv_spread"], color="purple")
ax.axhline(0, color="black", linewidth=0.8, linestyle="--")
ax.set_title("IV Spread: AAPL − SPY (ATM 30d) — Dispersion Premium")
ax.set_ylabel("IV Spread (pp)"); plt.tight_layout(); plt.show()

## 5. Greeks Over Time — θ, Γ, ν (ATM 30d)

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)
greek_labels = [("theta", "Theta (θ)", "steelblue"), ("gamma", "Gamma (Γ)", "green"), ("vega", "Vega (ν)", "red")]

for ax, (greek, label, color) in zip(axes, greek_labels):
    ax.plot(df_spy_atm_30["date"],  df_spy_atm_30[greek],  label=f"SPY {label}",  color=color, alpha=0.8)
    ax.plot(df_aapl_atm_30["date"], df_aapl_atm_30[greek], label=f"AAPL {label}", color=color, linestyle="--", alpha=0.8)
    ax.set_ylabel(label); ax.legend()
axes[0].set_title("Greeks Over Time — ATM 30d Options (SPY vs AAPL)")
plt.tight_layout(); plt.show()

## 6. Bid-Ask Spread Analysis

In [ ]:
for df, ticker in [(df_spy, "SPY"), (df_aapl, "AAPL")]:
    df = df[df["call_put"] == "C"].copy()
    df["spread"] = df["ask"] - df["bid"]
    df["rel_spread"] = df["spread"] / df["mid"].replace(0, np.nan)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Spread by moneyness bucket
    df["moneyness_bucket"] = pd.cut(df["moneyness"], bins=[0.7, 0.85, 0.95, 1.0, 1.05, 1.15, 1.3], labels=["0.7-0.85","0.85-0.95","0.95-1.0","1.0-1.05","1.05-1.15","1.15-1.3"])
    df.groupby("moneyness_bucket")["rel_spread"].median().plot(kind="bar", ax=axes[0], color="steelblue")
    axes[0].set_title(f"{ticker} — Relative Bid-Ask by Moneyness"); axes[0].set_ylabel("Median Rel. Spread")

    # Spread by DTE bucket
    df["dte_bucket"] = pd.cut(df["day_to_expiration"], bins=[0, 7, 14, 30, 60, 90, 365], labels=["0-7","7-14","14-30","30-60","60-90","90+"])
    df.groupby("dte_bucket")["rel_spread"].median().plot(kind="bar", ax=axes[1], color="darkorange")
    axes[1].set_title(f"{ticker} — Relative Bid-Ask by DTE"); axes[1].set_ylabel("Median Rel. Spread")

    plt.suptitle(f"{ticker} Bid-Ask Spread Analysis"); plt.tight_layout(); plt.show()

## 7. US Interest Rates

In [ ]:
rate_cols = ["1 Mo", "3 Mo", "6 Mo", "1 Yr", "2 Yr"]
rate_cols_present = [c for c in rate_cols if c in df_rates.columns]

fig, ax = plt.subplots(figsize=(14, 5))
for col in rate_cols_present:
    ax.plot(df_rates["date"], df_rates[col] * 100, label=col)
ax.set_title("US Treasury Par Yield Curve (2020–2022)")
ax.set_ylabel("Rate (%)"); ax.legend(); plt.tight_layout(); plt.show()

## 8. Summary

| Metric | SPY ATM 30d | AAPL ATM 30d |
|---|---|---|
| Mean IV | ... | ... |
| Std IV  | ... | ... |
| Mean θ  | ... | ... |
| Mean Γ  | ... | ... |
| Mean ν  | ... | ... |

In [ ]:
summary = pd.DataFrame({
    "SPY ATM 30d": df_spy_atm_30[["implied_volatility","theta","gamma","vega"]].mean(),
    "AAPL ATM 30d": df_aapl_atm_30[["implied_volatility","theta","gamma","vega"]].mean(),
}).T
summary.columns = ["Mean IV", "Mean θ", "Mean Γ", "Mean ν"]
summary["Mean IV"] = summary["Mean IV"] * 100
summary.round(4)